<a href="https://colab.research.google.com/github/VanshikaDhiman14/Computer-Vision-Labsheets/blob/main/Labsheet6cv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LABSHEET-6: IMAGE SEGMENTATION

1. Build a threshold-based segmentation tool with adjustable global and adaptive thresholds.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

uploaded=files.upload()
image_path=list(uploaded.keys())[0]
image=cv2.imread(image_path,cv2.IMREAD_GRAYSCALE)
if image is None:
  raise ValueError("Could not read the image.")

global_threshold=127
block_size=11
C=2

# global thresholding
_,global_binary=cv2.threshold(
    image,global_threshold,255,cv2.THRESH_BINARY
)
#Adaptive mean thresholding
adaptive_mean=cv2.adaptiveThreshold(
    image,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY,
    block_size,
    C
)
#Adaptive Gaussian thresholding
adaptive_gaussian=cv2.adaptiveThreshold(
    image,255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    block_size,
    C
)
plt.figure(figsize=(15, 8))

plt.subplot(2, 2, 1)
plt.imshow(image, cmap='gray')
plt.title("Original Grayscale Image")
plt.axis("off")

plt.subplot(2, 2, 2)
plt.imshow(global_binary, cmap='gray')
plt.title(f"Global Threshold = {global_threshold}")
plt.axis("off")

plt.subplot(2, 2, 3)
plt.imshow(adaptive_mean, cmap='gray')
plt.title(f"Adaptive Mean (Block={block_size}, C={C})")
plt.axis("off")

plt.subplot(2, 2, 4)
plt.imshow(adaptive_gaussian, cmap='gray')
plt.title(f"Adaptive Gaussian (Block={block_size}, C={C})")
plt.axis("off")

plt.tight_layout()
plt.show()

# -----------------------------
# Save results
# -----------------------------

cv2.imwrite("global_threshold.png", global_binary)
cv2.imwrite("adaptive_mean.png", adaptive_mean)
cv2.imwrite("adaptive_gaussian.png", adaptive_gaussian)

print("Segmentation completed successfully!")
print(f"Global threshold: {global_threshold}")
print(f"Adaptive block size: {block_size}")
print(f"Adaptive C value: {C}")
print("Vanshika Dhiman")

2. Develop a region-growing segmentation tool that segments an image starting from user-selected seed points.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from collections import deque
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

image = cv2.imread(image_path)

if image is None:
    raise ValueError("Could not read the image.")

# Convert BGR to RGB for displaying
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Select Seed Points
seed_points = []

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(image_rgb)
ax.set_title("Click on the image to select seed points.\n"
             "Close the image window after selecting points.")
ax.axis("off")

def select_seed(event):
    if event.xdata is not None and event.ydata is not None:
        x = int(event.xdata)
        y = int(event.ydata)

        seed_points.append((x, y))

        ax.plot(x, y, 'ro', markersize=8)
        ax.text(x + 5, y, str(len(seed_points)),
                color='red', fontsize=12)

        fig.canvas.draw()

fig.canvas.mpl_connect('button_press_event', select_seed)

plt.show()

# Check Seed Points
if len(seed_points) == 0:
    print("No seed points selected.")
    print("Please run the cell again and click on the image.")
else:
    print("Selected seed points:", seed_points)

    # ---------------------------------------------------
    # Region Growing Function
    # ---------------------------------------------------

    def region_growing(image, seeds, threshold=15):

        height, width = image.shape

        segmented = np.zeros((height, width), dtype=np.uint8)

        visited = np.zeros((height, width), dtype=bool)

        queue = deque()

        # Add all selected seed points
        for x, y in seeds:

            if 0 <= x < width and 0 <= y < height:

                queue.append((x, y))

                visited[y, x] = True
                segmented[y, x] = 255

        # 8-connected neighborhood
        neighbors = [
            (-1, -1), (0, -1), (1, -1),
            (-1,  0),          (1,  0),
            (-1,  1), (0,  1), (1,  1)
        ]

        # Grow the region
        while queue:

            x, y = queue.popleft()

            current_value = int(image[y, x])

            for dx, dy in neighbors:

                nx = x + dx
                ny = y + dy

                if 0 <= nx < width and 0 <= ny < height:

                    if not visited[ny, nx]:

                        neighbor_value = int(image[ny, nx])

                        # Similarity condition
                        if abs(neighbor_value - current_value) <= threshold:

                            visited[ny, nx] = True
                            segmented[ny, nx] = 255

                            queue.append((nx, ny))

        return segmented

    # ---------------------------------------------------
    # Adjustable Similarity Threshold
    # ---------------------------------------------------

    similarity_threshold = 15

    # Perform region growing
    segmented_image = region_growing(
        gray,
        seed_points,
        similarity_threshold
    )

    # ---------------------------------------------------
    # Display Results
    # ---------------------------------------------------

    plt.figure(figsize=(15, 6))

    plt.subplot(1, 3, 1)
    plt.imshow(image_rgb)
    plt.title("Original Image")
    plt.axis("off")

    # Show seed points
    for i, (x, y) in enumerate(seed_points):
        plt.plot(x, y, 'ro', markersize=8)
        plt.text(x + 5, y, str(i + 1),
                 color='red', fontsize=12)

    plt.subplot(1, 3, 2)
    plt.imshow(gray, cmap="gray")
    plt.title("Grayscale Image")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(segmented_image, cmap="gray")
    plt.title(
        f"Region Growing\nThreshold = {similarity_threshold}"
    )
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------
    # Save Segmented Image
    # ---------------------------------------------------

    cv2.imwrite("region_growing_result.png", segmented_image)

    print("Region growing segmentation completed!")
    print("Seed points:", seed_points)
    print("Similarity threshold:", similarity_threshold)

In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from collections import deque
from IPython.display import display, clear_output
import ipywidgets as widgets

# ---------------------------------------------------------
# 1. Upload Image
# ---------------------------------------------------------

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

image = cv2.imread(image_path)

if image is None:
    raise ValueError("Image could not be loaded.")

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

height, width = gray.shape

print("Image loaded successfully.")
print("Image size:", width, "x", height)

# ---------------------------------------------------------
# 2. Display Image
# ---------------------------------------------------------

plt.figure(figsize=(10, 7))
plt.imshow(image_rgb)
plt.title("IMAGE — Select your seed point coordinates below")
plt.axis("on")
plt.show()

# ---------------------------------------------------------
# 3. Enter Seed Coordinates
# ---------------------------------------------------------

print("\nSelect a point on the image.")
print("Use the displayed image coordinates:")
print("x = horizontal position")
print("y = vertical position")
print("\nYou can add multiple seed points.")

x_input = widgets.IntText(
    value=width // 2,
    description="X:",
    style={'description_width': 'initial'}
)

y_input = widgets.IntText(
    value=height // 2,
    description="Y:",
    style={'description_width': 'initial'}
)

add_button = widgets.Button(
    description="Add Seed Point",
    button_style="primary"
)

done_button = widgets.Button(
    description="Start Region Growing",
    button_style="success"
)

seed_points = []

output = widgets.Output()

# ---------------------------------------------------------
# 4. Add Seed Point
# ---------------------------------------------------------

def add_seed(b):

    x = x_input.value
    y = y_input.value

    if x < 0 or x >= width or y < 0 or y >= height:
        with output:
            print("Invalid coordinates!")
            print(f"X must be between 0 and {width-1}")
            print(f"Y must be between 0 and {height-1}")
            return

    seed_points.append((x, y))

    with output:
        clear_output(wait=True)
        print("Current seed points:")

        for i, point in enumerate(seed_points):
            print(f"Seed {i+1}: x={point[0]}, y={point[1]}")

add_button.on_click(add_seed)

# ---------------------------------------------------------
# 5. Region Growing Algorithm
# ---------------------------------------------------------

def region_growing(image, seeds, threshold):

    h, w = image.shape

    segmented = np.zeros((h, w), dtype=np.uint8)

    visited = np.zeros((h, w), dtype=bool)

    queue = deque()

    # Add seed points
    for x, y in seeds:

        queue.append((x, y))
        visited[y, x] = True
        segmented[y, x] = 255

    # 8-connected neighborhood
    neighbors = [
        (-1, -1), (0, -1), (1, -1),
        (-1,  0),          (1,  0),
        (-1,  1), (0,  1), (1,  1)
    ]

    while queue:

        x, y = queue.popleft()

        current_value = int(image[y, x])

        for dx, dy in neighbors:

            nx = x + dx
            ny = y + dy

            if 0 <= nx < w and 0 <= ny < h:

                if not visited[ny, nx]:

                    neighbor_value = int(image[ny, nx])

                    # Similarity condition
                    if abs(neighbor_value - current_value) <= threshold:

                        visited[ny, nx] = True
                        segmented[ny, nx] = 255

                        queue.append((nx, ny))

    return segmented

# ---------------------------------------------------------
# 6. Threshold Slider
# ---------------------------------------------------------

threshold_slider = widgets.IntSlider(
    value=15,
    min=1,
    max=50,
    step=1,
    description="Threshold:",
    style={'description_width': 'initial'}
)

# ---------------------------------------------------------
# 7. Start Segmentation
# ---------------------------------------------------------

def start_segmentation(b):

    if len(seed_points) == 0:

        with output:
            clear_output(wait=True)
            print("Please add at least one seed point first.")

        return

    threshold = threshold_slider.value

    segmented = region_growing(
        gray,
        seed_points,
        threshold
    )

    # Display result
    plt.figure(figsize=(15, 6))

    plt.subplot(1, 3, 1)
    plt.imshow(image_rgb)
    plt.title("Original Image")
    plt.axis("off")

    # Display seed points
    for i, (x, y) in enumerate(seed_points):

        plt.plot(
            x,
            y,
            'ro',
            markersize=8
        )

        plt.text(
            x + 5,
            y,
            str(i + 1),
            color='red',
            fontsize=12
        )

    plt.subplot(1, 3, 2)
    plt.imshow(gray, cmap="gray")
    plt.title("Grayscale Image")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(segmented, cmap="gray")
    plt.title(
        f"Region Growing\nThreshold = {threshold}"
    )
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    # Save result
    cv2.imwrite(
        "region_growing_result.png",
        segmented
    )

    with output:
        print("\nRegion growing completed!")
        print("Seed points:", seed_points)
        print("Threshold:", threshold)

# ---------------------------------------------------------
# 8. Display Controls
# ---------------------------------------------------------

display(
    x_input,
    y_input,
    add_button,
    threshold_slider,
    done_button,
    output
)

done_button.on_click(start_segmentation)

3. Create a split-and-merge segmentation demo on a textured or satellite image.

In [ ]:
# 3. Create a split-and-merge segmentation demo
#    on a textured or satellite image.

import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
image = cv2.imread("image.jpg")

if image is None:
    raise ValueError("Could not read the image.")

# Convert BGR -> RGB
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Use grayscale for segmentation
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Resize for faster processing
MAX_SIZE = 512

h, w = gray.shape

scale = min(MAX_SIZE / h, MAX_SIZE / w, 1)

if scale < 1:
    new_w = int(w * scale)
    new_h = int(h * scale)

    gray = cv2.resize(
        gray,
        (new_w, new_h)
    )

    image_rgb = cv2.resize(
        image_rgb,
        (new_w, new_h)
    )

# ============================================================
# 2. Adjustable Parameters
# ============================================================

# Maximum standard deviation allowed inside a region.
# Lower value = more splitting.
SPLIT_THRESHOLD = 15

# Difference between region means allowed during merging.
# Higher value = more merging.
MERGE_THRESHOLD = 10

# Minimum region size
MIN_SIZE = 8

# ============================================================
# 3. Split Stage
# ============================================================

regions = []

def split_region(x, y, width, height):

    region = gray[
        y:y + height,
        x:x + width
    ]

    # Avoid empty regions
    if region.size == 0:
        return

    mean = np.mean(region)
    std = np.std(region)

    # Stop if region is sufficiently homogeneous
    if (
        std <= SPLIT_THRESHOLD
        or width <= MIN_SIZE
        or height <= MIN_SIZE
    ):
        regions.append(
            (x, y, width, height, mean)
        )
        return

    # Split into four quadrants
    half_w = width // 2
    half_h = height // 2

    # Prevent zero-sized regions
    if half_w == 0 or half_h == 0:
        regions.append(
            (x, y, width, height, mean)
        )
        return

    split_region(
        x,
        y,
        half_w,
        half_h
    )

    split_region(
        x + half_w,
        y,
        width - half_w,
        half_h
    )

    split_region(
        x,
        y + half_h,
        half_w,
        height - half_h
    )

    split_region(
        x + half_w,
        y + half_h,
        width - half_w,
        height - half_h
    )


# Start splitting
H, W = gray.shape

split_region(
    0,
    0,
    W,
    H
)

print("Number of regions after splitting:", len(regions))

# ============================================================
# 4. Create Split Visualization
# ============================================================

split_image = image_rgb.copy()

for x, y, width, height, mean in regions:

    cv2.rectangle(
        split_image,
        (x, y),
        (x + width, y + height),
        (255, 0, 0),
        1
    )

# ============================================================
# 5. Merge Similar Neighboring Regions
# ============================================================

parent = list(range(len(regions)))

def find(x):

    while parent[x] != x:

        parent[x] = parent[parent[x]]
        x = parent[x]

    return x


def union(a, b):

    root_a = find(a)
    root_b = find(b)

    if root_a != root_b:
        parent[root_b] = root_a


def are_neighbors(r1, r2):

    x1, y1, w1, h1, _ = r1
    x2, y2, w2, h2, _ = r2

    # Horizontal neighbors
    horizontal = (
        (x1 + w1 == x2 or x2 + w2 == x1)
        and
        max(y1, y2) < min(y1 + h1, y2 + h2)
    )

    # Vertical neighbors
    vertical = (
        (y1 + h1 == y2 or y2 + h2 == y1)
        and
        max(x1, x2) < min(x1 + w1, x2 + w2)
    )

    return horizontal or vertical


# Compare neighboring regions
for i in range(len(regions)):

    for j in range(i + 1, len(regions)):

        if are_neighbors(
            regions[i],
            regions[j]
        ):

            mean1 = regions[i][4]
            mean2 = regions[j][4]

            if abs(mean1 - mean2) <= MERGE_THRESHOLD:
                union(i, j)

# ============================================================
# 6. Create Final Segmentation
# ============================================================

labels = {}

label_count = 0

for i in range(len(regions)):

    root = find(i)

    if root not in labels:
        labels[root] = label_count
        label_count += 1


segmented = np.zeros(
    (H, W),
    dtype=np.uint8
)

# Generate different grayscale labels
for i, region in enumerate(regions):

    x, y, width, height, mean = region

    label = labels[find(i)]

    # Give each merged region a different intensity
    intensity = int(
        (label * 255) / max(label_count - 1, 1)
    )

    segmented[
        y:y + height,
        x:x + width
    ] = intensity

print("Number of regions after merging:", label_count)

# ============================================================
# 7. Display Results
# ============================================================

plt.figure(figsize=(18, 6))

# Original
plt.subplot(1, 3, 1)
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")

# Split result
plt.subplot(1, 3, 2)
plt.imshow(split_image)
plt.title(
    f"Split Stage\nRegions = {len(regions)}"
)
plt.axis("off")

# Final result
plt.subplot(1, 3, 3)
plt.imshow(segmented, cmap="nipy_spectral")
plt.title(
    f"Final Split-and-Merge\nRegions = {label_count}"
)
plt.axis("off")

plt.tight_layout()
plt.show()

# ============================================================
# 8. Save Result
# ============================================================

cv2.imwrite(
    "split_merge_segmentation.png",
    segmented
)

print("\nSegmentation completed successfully!")
print("Saved as: split_merge_segmentation.png")
print("Vanshika Dhiman")

4. Build a watershed segmentation tool to separate touching or overlapping objects(eg. coins or cells).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

image = cv2.imread("/content/img6.png")

if image is None:
    raise ValueError("Could not read the image.")

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# ============================================================
# 2. Convert to Grayscale
# ============================================================

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Reduce noise
blur = cv2.GaussianBlur(
    gray,
    (5, 5),
    0
)

# ============================================================
# 3. Create Binary Image
# ============================================================

_, binary = cv2.threshold(
    blur,
    0,
    255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

# If foreground appears mostly black, invert it
if np.sum(binary == 255) < np.sum(binary == 0):
    binary = cv2.bitwise_not(binary)

# ============================================================
# 4. Morphological Opening
# ============================================================

kernel = np.ones((3, 3), np.uint8)

opening = cv2.morphologyEx(
    binary,
    cv2.MORPH_OPEN,
    kernel,
    iterations=2
)

# ============================================================
# 5. Determine Sure Background
# ============================================================

sure_bg = cv2.dilate(
    opening,
    kernel,
    iterations=3
)

# ============================================================
# 6. Distance Transform
# ============================================================

dist_transform = cv2.distanceTransform(
    opening,
    cv2.DIST_L2,
    5
)

# ============================================================
# 7. Determine Sure Foreground
# ============================================================

# Change this value if objects are not detected correctly
foreground_ratio = 0.45

_, sure_fg = cv2.threshold(
    dist_transform,
    foreground_ratio * dist_transform.max(),
    255,
    0
)

sure_fg = np.uint8(sure_fg)

# ============================================================
# 8. Unknown Region
# ============================================================

unknown = cv2.subtract(
    sure_bg,
    sure_fg
)

# ============================================================
# 9. Create Markers
# ============================================================

num_markers, markers = cv2.connectedComponents(
    sure_fg
)

# Add 1 so that background becomes marker 1
markers = markers + 1

# Mark unknown region as 0
markers[unknown == 255] = 0

print("Initial markers:", num_markers)

# ============================================================
# 10. Apply Watershed
# ============================================================

markers = cv2.watershed(
    image,
    markers
)

# Watershed boundaries are marked with -1
boundary = image_rgb.copy()

boundary[markers == -1] = [255, 0, 0]

# ============================================================
# 11. Create Colored Segmentation
# ============================================================

segmented = np.zeros_like(image_rgb)

unique_labels = np.unique(markers)

# Generate different colors for each detected object
np.random.seed(42)

colors = {}

for label in unique_labels:

    if label <= 1:
        continue

    colors[label] = np.random.randint(
        0,
        256,
        3
    )

for label, color in colors.items():

    segmented[markers == label] = color

# Draw watershed boundaries
segmented[markers == -1] = [255, 255, 255]

# ============================================================
# 12. Display Results
# ============================================================

plt.figure(figsize=(18, 10))

plt.subplot(2, 3, 1)
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(binary, cmap="gray")
plt.title("Binary Image")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(dist_transform, cmap="gray")
plt.title("Distance Transform")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(sure_fg, cmap="gray")
plt.title("Sure Foreground")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(boundary)
plt.title("Watershed Boundaries")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(segmented)
plt.title("Final Segmentation")
plt.axis("off")

plt.tight_layout()
plt.show()

# ============================================================
# 13. Count Detected Objects
# ============================================================

object_labels = unique_labels[unique_labels > 1]

print("Objects detected:", len(object_labels))

# ============================================================
# 14. Save Result
# ============================================================

cv2.imwrite(
    "watershed_segmentation.png",
    cv2.cvtColor(segmented, cv2.COLOR_RGB2BGR)
)

print("Watershed segmentation completed!")
print("Result saved as: watershed_segmentation.png")
print("Vanshika Dhiman")

5. Develop a K-means or mean-shift color segmentation tool that  clusters image regions by color similarity.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
image = cv2.imread("image.jpg")

if image is None:
    raise ValueError("Could not read the image.")

# Convert BGR to RGB
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# ============================================================
# 2. Resize Image
# ============================================================

# Resizing makes K-Means faster for large images
scale = 0.5

small_image = cv2.resize(
    image_rgb,
    None,
    fx=scale,
    fy=scale
)

# ============================================================
# 3. Convert Image to Pixel Data
# ============================================================

# Reshape image into a list of pixels
pixels = small_image.reshape(
    (-1, 3)
)

# Convert to float32 for K-Means
pixels = np.float32(pixels)

# ============================================================
# 4. Adjustable K-Means Parameters
# ============================================================

# Number of color clusters
K = 5

# Stopping criteria
criteria = (
    cv2.TERM_CRITERIA_EPS +
    cv2.TERM_CRITERIA_MAX_ITER,
    100,
    0.2
)

# Multiple attempts with different initial centers
attempts = 10

# ============================================================
# 5. Apply K-Means
# ============================================================

compactness, labels, centers = cv2.kmeans(
    pixels,
    K,
    None,
    criteria,
    attempts,
    cv2.KMEANS_PP_CENTERS
)

# Convert cluster centers back to uint8
centers = np.uint8(centers)

# ============================================================
# 6. Reconstruct Segmented Image
# ============================================================

labels = labels.flatten()

segmented_pixels = centers[labels]

segmented_image = segmented_pixels.reshape(
    small_image.shape
)

# ============================================================
# 7. Display Cluster Information
# ============================================================

print("K-Means Color Segmentation")
print("--------------------------------")
print("Number of clusters (K):", K)
print("Image size:", small_image.shape[1], "x", small_image.shape[0])

print("\nCluster Colors:")

for i, center in enumerate(centers):

    print(
        f"Cluster {i + 1}: "
        f"R={center[0]}, "
        f"G={center[1]}, "
        f"B={center[2]}"
    )

# ============================================================
# 8. Display Results
# ============================================================

plt.figure(figsize=(15, 7))

plt.subplot(1, 2, 1)
plt.imshow(small_image)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(segmented_image)
plt.title(f"K-Means Segmentation (K={K})")
plt.axis("off")

plt.tight_layout()
plt.show()

# ============================================================
# 9. Display Individual Clusters
# ============================================================

plt.figure(figsize=(15, 8))

for i in range(K):

    # Create mask for current cluster
    mask = labels == i

    # Create black image
    cluster_image = np.zeros_like(small_image)

    # Keep only pixels belonging to this cluster
    cluster_image.reshape(-1, 3)[mask] = centers[i]

    plt.subplot(
        2,
        int(np.ceil(K / 2)),
        i + 1
    )

    plt.imshow(cluster_image)
    plt.title(f"Cluster {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

# ============================================================
# 10. Save Segmented Image
# ============================================================

cv2.imwrite(
    "kmeans_color_segmentation.png",
    cv2.cvtColor(
        segmented_image,
        cv2.COLOR_RGB2BGR
    )
)

print("\nSegmentation completed successfully!")
print("Saved as: kmeans_color_segmentation.png")
print("Vanshika Dhiman")

6.Create an interactive foreground-extraction tool using a graph nased method such as GrabCut.

In [ ]:
# 6. Create an interactive foreground-extraction tool
#    using GrabCut.

import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display, clear_output
import ipywidgets as widgets

image = cv2.imread("img6.png")

if image is None:
    raise ValueError("Could not read the image.")

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

height, width = image.shape[:2]

print("Image loaded successfully!")
print("Image size:", width, "x", height)

# ============================================================
# 2. Display Image and Get Rectangle Coordinates
# ============================================================

plt.figure(figsize=(10, 7))
plt.imshow(image_rgb)
plt.title(
    "Draw an imaginary rectangle around the object.\n"
    "Enter its X, Y, Width and Height below."
)
plt.axis("on")
plt.show()

# ============================================================
# 3. Rectangle Controls
# ============================================================

x_input = widgets.IntText(
    value=int(width * 0.2),
    description="X:",
    style={'description_width': 'initial'}
)

y_input = widgets.IntText(
    value=int(height * 0.2),
    description="Y:",
    style={'description_width': 'initial'}
)

w_input = widgets.IntText(
    value=int(width * 0.6),
    description="Width:",
    style={'description_width': 'initial'}
)

h_input = widgets.IntText(
    value=int(height * 0.6),
    description="Height:",
    style={'description_width': 'initial'}
)

run_button = widgets.Button(
    description="Extract Foreground",
    button_style="success"
)

output = widgets.Output()

display(
    x_input,
    y_input,
    w_input,
    h_input,
    run_button,
    output
)

# ============================================================
# 4. GrabCut Function
# ============================================================

def run_grabcut(button):

    with output:

        clear_output(wait=True)

        # Get rectangle values
        x = x_input.value
        y = y_input.value
        rect_width = w_input.value
        rect_height = h_input.value

        # ----------------------------------------------------
        # Validate rectangle
        # ----------------------------------------------------

        if x < 0 or y < 0:
            print("X and Y cannot be negative.")
            return

        if rect_width <= 0 or rect_height <= 0:
            print("Width and height must be greater than zero.")
            return

        if x + rect_width > width:
            print("Rectangle extends outside the image width.")
            return

        if y + rect_height > height:
            print("Rectangle extends outside the image height.")
            return

        # ----------------------------------------------------
        # Create GrabCut mask
        # ----------------------------------------------------

        mask = np.zeros(
            image.shape[:2],
            np.uint8
        )

        # Background and foreground models
        bgd_model = np.zeros(
            (1, 65),
            np.float64
        )

        fgd_model = np.zeros(
            (1, 65),
            np.float64
        )

        # ----------------------------------------------------
        # Apply GrabCut
        # ----------------------------------------------------

        cv2.grabCut(
            image,
            mask,
            (x, y, rect_width, rect_height),
            bgd_model,
            fgd_model,
            5,
            cv2.GC_INIT_WITH_RECT
        )

        # ----------------------------------------------------
        # Create foreground mask
        # ----------------------------------------------------

        foreground_mask = np.where(
            (mask == cv2.GC_FGD) |
            (mask == cv2.GC_PR_FGD),
            255,
            0
        ).astype("uint8")

        # ----------------------------------------------------
        # Extract foreground
        # ----------------------------------------------------

        foreground = cv2.bitwise_and(
            image,
            image,
            mask=foreground_mask
        )

        foreground_rgb = cv2.cvtColor(
            foreground,
            cv2.COLOR_BGR2RGB
        )

        # ----------------------------------------------------
        # Create transparent background
        # ----------------------------------------------------

        transparent = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2BGRA
        )

        transparent[:, :, 3] = foreground_mask

        transparent_rgb = cv2.cvtColor(
            transparent,
            cv2.COLOR_BGRA2RGBA
        )

        # ----------------------------------------------------
        # Display results
        # ----------------------------------------------------

        plt.figure(figsize=(18, 6))

        # Original
        plt.subplot(1, 3, 1)

        display_image = image_rgb.copy()

        cv2.rectangle(
            display_image,
            (x, y),
            (x + rect_width, y + rect_height),
            (255, 0, 0),
            3
        )

        plt.imshow(display_image)
        plt.title("Selected Rectangle")
        plt.axis("off")

        # Mask
        plt.subplot(1, 3, 2)

        plt.imshow(
            foreground_mask,
            cmap="gray"
        )

        plt.title("GrabCut Foreground Mask")
        plt.axis("off")

        # Extracted foreground
        plt.subplot(1, 3, 3)

        plt.imshow(foreground_rgb)
        plt.title("Extracted Foreground")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

        # ----------------------------------------------------
        # Save results
        # ----------------------------------------------------

        cv2.imwrite(
            "grabcut_foreground.png",
            foreground
        )

        cv2.imwrite(
            "grabcut_mask.png",
            foreground_mask
        )

        cv2.imwrite(
            "grabcut_transparent.png",
            transparent
        )

        print("GrabCut extraction completed!")
        print("Foreground saved as: grabcut_foreground.png")
        print("Mask saved as: grabcut_mask.png")
        print("Transparent result saved as: grabcut_transparent.png")


# Connect button
run_button.on_click(run_grabcut)
print("Vanshika Dhiman")

7. Build a normalised cut based segmentation demo that partitions an image into meaningful regions.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import SpectralClustering
from google.colab import files

image = cv2.imread("img6.png")

if image is None:
    raise ValueError("Could not read the uploaded image.")

# Convert BGR to RGB
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Resize for faster processing
max_size = 150
h, w = image_rgb.shape[:2]

scale = min(max_size / h, max_size / w, 1)

new_w = int(w * scale)
new_h = int(h * scale)

image_small = cv2.resize(image_rgb, (new_w, new_h))

# ---------------------------------------------------------
# 2. Prepare Feature Data
# ---------------------------------------------------------
h, w, c = image_small.shape

# Get pixel coordinates
Y, X = np.mgrid[0:h, 0:w]

# Normalize coordinates
X = X / w
Y = Y / h

# Normalize RGB values
pixels = image_small.reshape(-1, 3) / 255.0

# Combine color and spatial information
# This helps create meaningful spatial regions.
features = np.column_stack([
    pixels[:, 0],
    pixels[:, 1],
    pixels[:, 2],
    X.flatten(),
    Y.flatten()
])

# ---------------------------------------------------------
# 3. Normalized Cut / Spectral Clustering
# ---------------------------------------------------------
N_CLUSTERS = 4

print("Applying Normalized Cut based segmentation...")
print("Number of regions:", N_CLUSTERS)

# Spectral clustering approximates the normalized cut
model = SpectralClustering(
    n_clusters=N_CLUSTERS,
    affinity='nearest_neighbors',
    n_neighbors=10,
    assign_labels='kmeans',
    random_state=42
)

labels = model.fit_predict(features)

# Reshape labels into image
segmentation = labels.reshape(h, w)

# ---------------------------------------------------------
# 4. Create Colored Segmentation
# ---------------------------------------------------------
segmented_image = np.zeros_like(image_small)

# Generate different colors for each region
colors = [
    [255, 0, 0],
    [0, 255, 0],
    [0, 0, 255],
    [255, 255, 0],
    [255, 0, 255],
    [0, 255, 255],
    [255, 128, 0],
    [128, 0, 255]
]

for i in range(N_CLUSTERS):
    segmented_image[segmentation == i] = colors[i % len(colors)]

# ---------------------------------------------------------
# 5. Create Region Mean-Color Image
# ---------------------------------------------------------
mean_segmented = np.zeros_like(image_small)

for i in range(N_CLUSTERS):
    mask = segmentation == i

    if np.any(mask):
        mean_color = image_small[mask].mean(axis=0)
        mean_segmented[mask] = mean_color.astype(np.uint8)

# ---------------------------------------------------------
# 6. Display Results
# ---------------------------------------------------------
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image_small)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(segmented_image)
plt.title("Normalized Cut Regions")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(mean_segmented)
plt.title("Mean-Color Segmentation")
plt.axis("off")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 7. Save Results
# ---------------------------------------------------------
cv2.imwrite(
    "normalized_cut_segmentation.png",
    cv2.cvtColor(segmented_image, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    "normalized_cut_mean_color.png",
    cv2.cvtColor(mean_segmented, cv2.COLOR_RGB2BGR)
)

print("Segmentation completed!")
print("Saved:")
print("1. normalized_cut_segmentation.png")
print("2. normalized_cut_mean_color.png")
print("Vanshika Dhiman")

8.Develop a medical image segmentation tool(e.g. isolating a tumour or organ outline from  a grayscale scan).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

image = cv2.imread("img7.jfif", cv2.IMREAD_GRAYSCALE)

if image is None:
    raise ValueError("Could not read the uploaded image.")

# ---------------------------------------------------------
# 2. Resize Image for Faster Processing
# ---------------------------------------------------------
max_size = 512

h, w = image.shape
scale = min(max_size / h, max_size / w, 1)

new_w = int(w * scale)
new_h = int(h * scale)

image = cv2.resize(image, (new_w, new_h))

# ---------------------------------------------------------
# 3. Noise Reduction
# ---------------------------------------------------------
blurred = cv2.GaussianBlur(image, (5, 5), 0)

# ---------------------------------------------------------
# 4. Otsu Thresholding
# ---------------------------------------------------------
threshold_value, binary = cv2.threshold(
    blurred,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

print("Automatically selected threshold:", threshold_value)

# ---------------------------------------------------------
# 5. Morphological Operations
# ---------------------------------------------------------
kernel = np.ones((5, 5), np.uint8)

# Remove small noise
cleaned = cv2.morphologyEx(
    binary,
    cv2.MORPH_OPEN,
    kernel,
    iterations=2
)

# Fill small gaps
cleaned = cv2.morphologyEx(
    cleaned,
    cv2.MORPH_CLOSE,
    kernel,
    iterations=2
)

# ---------------------------------------------------------
# 6. Find Connected Components
# ---------------------------------------------------------
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    cleaned,
    connectivity=8
)

# Create empty mask
segmentation = np.zeros_like(cleaned)

# Keep sufficiently large regions
MIN_AREA = 500

for i in range(1, num_labels):
    area = stats[i, cv2.CC_STAT_AREA]

    if area >= MIN_AREA:
        segmentation[labels == i] = 255

# ---------------------------------------------------------
# 7. Create Segmentation Overlay
# ---------------------------------------------------------
image_color = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

# Create red overlay
overlay = image_color.copy()

overlay[segmentation == 255] = [255, 0, 0]

# Blend original and segmentation
result = cv2.addWeighted(
    image_color,
    0.7,
    overlay,
    0.3,
    0
)

# ---------------------------------------------------------
# 8. Draw Contours
# ---------------------------------------------------------
contours, _ = cv2.findContours(
    segmentation,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

contour_image = image_color.copy()

cv2.drawContours(
    contour_image,
    contours,
    -1,
    (0, 255, 0),
    2
)

# ---------------------------------------------------------
# 9. Display Results
# ---------------------------------------------------------
plt.figure(figsize=(16, 8))

plt.subplot(2, 3, 1)
plt.imshow(image, cmap="gray")
plt.title("Original Medical Scan")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(blurred, cmap="gray")
plt.title("Noise Reduced")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(binary, cmap="gray")
plt.title("Otsu Threshold")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(segmentation, cmap="gray")
plt.title("Segmented Region")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(contour_image)
plt.title("Detected Region Boundary")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(result)
plt.title("Segmentation Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 10. Save Results
# ---------------------------------------------------------
cv2.imwrite(
    "medical_segmentation_mask.png",
    segmentation
)

cv2.imwrite(
    "medical_segmentation_contours.png",
    cv2.cvtColor(contour_image, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    "medical_segmentation_overlay.png",
    cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
)

print("\nSegmentation completed successfully!")
print("Saved files:")
print("- medical_segmentation_mask.png")
print("- medical_segmentation_contours.png")
print("- medical_segmentation_overlay.png")

9. Create a license plate or text region segmentation tool that isolates text areas from vechile or document images.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

image = cv2.imread("charminar2.jpg")

if image is None:
    raise ValueError("Could not read the uploaded image.")

# Convert BGR to RGB for displaying
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# ---------------------------------------------------------
# 2. Resize Image
# ---------------------------------------------------------
max_size = 800

h, w = image.shape[:2]
scale = min(max_size / h, max_size / w, 1)

new_w = int(w * scale)
new_h = int(h * scale)

image = cv2.resize(image, (new_w, new_h))
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# ---------------------------------------------------------
# 3. Convert to Grayscale
# ---------------------------------------------------------
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# ---------------------------------------------------------
# 4. Noise Reduction
# ---------------------------------------------------------
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

# ---------------------------------------------------------
# 5. Adaptive Thresholding
# ---------------------------------------------------------
binary = cv2.adaptiveThreshold(
    blurred,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV,
    21,
    10
)

# ---------------------------------------------------------
# 6. Morphological Operation
# ---------------------------------------------------------
# Horizontal kernel helps connect characters into text regions
kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (25, 5)
)

morph = cv2.morphologyEx(
    binary,
    cv2.MORPH_CLOSE,
    kernel
)

# Remove small noise
small_kernel = np.ones((3, 3), np.uint8)

morph = cv2.morphologyEx(
    morph,
    cv2.MORPH_OPEN,
    small_kernel
)

# ---------------------------------------------------------
# 7. Find Contours
# ---------------------------------------------------------
contours, _ = cv2.findContours(
    morph,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

# Copy image for drawing detected regions
result = image_rgb.copy()

# Mask for extracted text regions
text_mask = np.zeros(gray.shape, dtype=np.uint8)

detected_regions = 0

# ---------------------------------------------------------
# 8. Filter Text-Like Regions
# ---------------------------------------------------------
for contour in contours:

    x, y, w_box, h_box = cv2.boundingRect(contour)

    area = w_box * h_box

    if area < 500:
        continue

    aspect_ratio = w_box / float(h_box)

    # Text regions are generally wider than they are tall
    if 1.5 <= aspect_ratio <= 15:

        # Draw rectangle
        cv2.rectangle(
            result,
            (x, y),
            (x + w_box, y + h_box),
            (255, 0, 0),
            2
        )

        # Add region to mask
        cv2.rectangle(
            text_mask,
            (x, y),
            (x + w_box, y + h_box),
            255,
            -1
        )

        detected_regions += 1

# ---------------------------------------------------------
# 9. Extract Text Regions
# ---------------------------------------------------------
text_regions = cv2.bitwise_and(
    image_rgb,
    image_rgb,
    mask=text_mask
)

# ---------------------------------------------------------
# 10. Display Results
# ---------------------------------------------------------
plt.figure(figsize=(16, 10))

plt.subplot(2, 3, 1)
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(gray, cmap="gray")
plt.title("Grayscale")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(binary, cmap="gray")
plt.title("Adaptive Threshold")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(morph, cmap="gray")
plt.title("Morphological Processing")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(result)
plt.title("Detected Text Regions")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(text_regions)
plt.title("Extracted Text Areas")
plt.axis("off")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 11. Print Results
# ---------------------------------------------------------
print("Number of text-like regions detected:", detected_regions)

# ---------------------------------------------------------
# 12. Save Results
# ---------------------------------------------------------
cv2.imwrite(
    "text_region_mask.png",
    text_mask
)

cv2.imwrite(
    "detected_text_regions.png",
    cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
)

cv2.imwrite(
    "extracted_text_areas.png",
    cv2.cvtColor(text_regions, cv2.COLOR_RGB2BGR)
)

print("\nSaved files:")
print("1. text_region_mask.png")
print("2. detected_text_regions.png")
print("3. extracted_text_areas.png")
print("Vanshika Dhiman")

10. Build an evaluation dashboard that compares segmentation accuracy(IoU/Dice score) of different methods against ground-truth masks.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import files
image = cv2.imread("charminar1.jpg")

if image is None:
    raise ValueError("Could not read the original image.")


ground_truth = cv2.imread("charminar2.jpg", cv2.IMREAD_GRAYSCALE)

if ground_truth is None:
    raise ValueError("Could not read the ground-truth mask.")

# =========================================================
# 2. Resize Ground Truth to Image Size
# =========================================================

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

ground_truth = cv2.resize(
    ground_truth,
    (gray.shape[1], gray.shape[0])
)

# Convert ground truth to binary
_, ground_truth = cv2.threshold(
    ground_truth,
    127,
    255,
    cv2.THRESH_BINARY
)

# =========================================================
# 3. Segmentation Method 1 - Otsu Thresholding
# =========================================================

_, otsu_mask = cv2.threshold(
    gray,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# =========================================================
# 4. Segmentation Method 2 - Adaptive Thresholding
# =========================================================

adaptive_mask = cv2.adaptiveThreshold(
    gray,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    21,
    5
)

# =========================================================
# 5. Segmentation Method 3 - Edge + Morphology
# =========================================================

edges = cv2.Canny(gray, 50, 150)

kernel = np.ones((5, 5), np.uint8)

edge_mask = cv2.dilate(
    edges,
    kernel,
    iterations=2
)

edge_mask = cv2.morphologyEx(
    edge_mask,
    cv2.MORPH_CLOSE,
    kernel
)

# =========================================================
# 6. Segmentation Method 4 - K-Means
# =========================================================

pixels = gray.reshape((-1, 1)).astype(np.float32)

criteria = (
    cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
    100,
    0.2
)

K = 2

_, labels, centers = cv2.kmeans(
    pixels,
    K,
    None,
    criteria,
    10,
    cv2.KMEANS_RANDOM_CENTERS
)

centers = np.uint8(centers)

kmeans_mask = centers[labels.flatten()].reshape(gray.shape)

# Convert to binary
_, kmeans_mask = cv2.threshold(
    kmeans_mask,
    127,
    255,
    cv2.THRESH_BINARY
)

# =========================================================
# 7. Make All Masks Binary
# =========================================================

masks = {
    "Otsu": otsu_mask,
    "Adaptive": adaptive_mask,
    "Edge + Morphology": edge_mask,
    "K-Means": kmeans_mask
}

# =========================================================
# 8. IoU Calculation
# =========================================================

def calculate_iou(predicted, ground_truth):

    predicted = predicted > 0
    ground_truth = ground_truth > 0

    intersection = np.logical_and(
        predicted,
        ground_truth
    ).sum()

    union = np.logical_or(
        predicted,
        ground_truth
    ).sum()

    if union == 0:
        return 1.0

    return intersection / union


# =========================================================
# 9. Dice Score Calculation
# =========================================================

def calculate_dice(predicted, ground_truth):

    predicted = predicted > 0
    ground_truth = ground_truth > 0

    intersection = np.logical_and(
        predicted,
        ground_truth
    ).sum()

    total_pixels = predicted.sum() + ground_truth.sum()

    if total_pixels == 0:
        return 1.0

    return (2 * intersection) / total_pixels


# =========================================================
# 10. Evaluate All Methods
# =========================================================

results = []

for method_name, mask in masks.items():

    iou = calculate_iou(
        mask,
        ground_truth
    )

    dice = calculate_dice(
        mask,
        ground_truth
    )

    results.append({
        "Method": method_name,
        "IoU": iou,
        "Dice Score": dice
    })

# Create DataFrame
results_df = pd.DataFrame(results)

# Sort by Dice Score
results_df = results_df.sort_values(
    by="Dice Score",
    ascending=False
)

# =========================================================
# 11. Display Evaluation Table
# =========================================================

print("\n========== SEGMENTATION EVALUATION ==========\n")

display(
    results_df.style.format({
        "IoU": "{:.4f}",
        "Dice Score": "{:.4f}"
    })
)

# =========================================================
# 12. Display Segmentation Results
# =========================================================

plt.figure(figsize=(16, 10))

plt.subplot(2, 3, 1)
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Original Image")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(ground_truth, cmap="gray")
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(otsu_mask, cmap="gray")
plt.title("Otsu")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(adaptive_mask, cmap="gray")
plt.title("Adaptive Threshold")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(edge_mask, cmap="gray")
plt.title("Edge + Morphology")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(kmeans_mask, cmap="gray")
plt.title("K-Means")
plt.axis("off")

plt.tight_layout()
plt.show()

# =========================================================
# 13. Create IoU and Dice Comparison Chart
# =========================================================

methods = results_df["Method"].values
iou_scores = results_df["IoU"].values
dice_scores = results_df["Dice Score"].values

x = np.arange(len(methods))
width = 0.35

plt.figure(figsize=(12, 6))

plt.bar(
    x - width/2,
    iou_scores,
    width,
    label="IoU"
)

plt.bar(
    x + width/2,
    dice_scores,
    width,
    label="Dice Score"
)

plt.xlabel("Segmentation Method")
plt.ylabel("Score")
plt.title("Segmentation Accuracy Comparison")

plt.xticks(
    x,
    methods,
    rotation=20
)

plt.ylim(0, 1)
plt.legend()
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# =========================================================
# 14. Display Best Method
# =========================================================

best_method = results_df.iloc[0]["Method"]
best_dice = results_df.iloc[0]["Dice Score"]

print("\n==============================================")
print("BEST SEGMENTATION METHOD")
print("==============================================")
print("Method:", best_method)
print("Dice Score:", round(best_dice, 4))
print("IoU:", round(
    results_df.iloc[0]["IoU"],
    4
))

# =========================================================
# 15. Save Evaluation Results
# =========================================================

results_df.to_csv(
    "segmentation_evaluation_results.csv",
    index=False
)

print("\nEvaluation results saved as:")
print("segmentation_evaluation_results.csv")
print("Vanshika Dhiman")